In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt


In [22]:
givenWkdir = ''
df_optimumK=pd.read_csv("BestKvalueCluster_VT.csv")
df_ssdeepClusters=pd.read_csv("/home/mabon/research/Autoyara/YaraResults/clusterCSV/ssdeep/th60.csv")
df_ssdeepClusters['File_Path'] = df_ssdeepClusters['File_Path'].str.replace('/usr/src/app/', givenWkdir, regex=False)
df_vtfam=pd.read_csv("/home/mabon/research/Autoyara/YaraResults/clusterCSV/virusTotal/MainVtCluster.csv")
df_vtfam['File_Path'] = df_vtfam['File_Path'].str.replace('/mnt/data_disk1/mabon/', givenWkdir, regex=False)
unique_paths_b = df_vtfam['File_Path'].unique()
df_ssdeepClusters=df_ssdeepClusters[df_ssdeepClusters['File_Path'].isin(unique_paths_b)]

In [23]:
from tqdm import tqdm

# Step 2: Function to extract file name from File_Path
def extract_file_name(file_path):
    return os.path.basename(file_path)

# Initialize lists to store new DataFrame data
result_data = {
    'File_Name': [],
    'Cluster_Label_A': [],
    'Cluster_Label_B': [],
    'File_Path_A': [],
    'File_Path_B': []
}

# Group File_Path by Cluster_Label in Dataframe A
cluster_paths = df_ssdeepClusters.groupby('Cluster_Label')['File_Path'].apply(list).to_dict()

# Iterate through each Cluster_Label and File_Path in Dataframe A
for cluster_label, paths in tqdm(cluster_paths.items(), desc="Processing Clusters"):
    for path in paths:
        file_name = extract_file_name(path)
        # Find matching row in Dataframe B
        matching_row = df_vtfam[df_vtfam['File_Name'] == file_name]
        if not matching_row.empty:
            # Match found, append data
            result_data['File_Name'].append(file_name)
            result_data['Cluster_Label_A'].append(cluster_label)
            result_data['Cluster_Label_B'].append(matching_row['Cluster_Label'].iloc[0])
            result_data['File_Path_A'].append(path)
            result_data['File_Path_B'].append(matching_row['File_Path'].iloc[0])
        else:
            # No match, append with None for Dataframe B columns
            result_data['File_Name'].append(file_name)
            result_data['Cluster_Label_A'].append(cluster_label)
            result_data['Cluster_Label_B'].append(None)
            result_data['File_Path_A'].append(path)
            result_data['File_Path_B'].append(None)
# Create new DataFrame
df_result = pd.DataFrame(result_data)


Processing Clusters: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30151/30151 [02:01<00:00, 248.91it/s]


In [24]:
df=df_result

In [25]:
unique_labels = df_result.groupby('Cluster_Label_A')['Cluster_Label_B'].unique().reset_index()

# Store Cluster_Label_B values as a list in unique_identity
unique_labels['unique_identity'] = unique_labels['Cluster_Label_B'].apply(lambda x: list(x))

# Merge the unique_identity back to the original dataframe based on Cluster_Label_A
df_result = df_result.merge(unique_labels[['Cluster_Label_A', 'unique_identity']], on='Cluster_Label_A', how='left')


In [26]:
# Step 1: Create mapping from cluster_number to best_k_value
mapping = dict(zip(df_optimumK['cluster_number'], df_optimumK['best_k_value']))

# Step 2: Replace each cluster in unique_identity list with its best_k_value
def map_cluster_list(cluster_list):
    return [mapping.get(cluster, cluster) for cluster in cluster_list]  # fallback to original if not in mapping

# Step 3: Apply transformation
df_result['unique_identity'] = df_result['unique_identity'].apply(map_cluster_list)


In [27]:

# # Calculate cluster size for Cluster_Label_A
# cluster_a_sizes = df['Cluster_Label_A'].value_counts().reset_index()
# cluster_a_sizes.columns = ['Cluster_Label_A', 'Cluster_Size']

# # Group by Cluster_Label_A and Cluster_Label_B to get unique best_k_value counts
# k_value_counts = df.groupby('Cluster_Label_A')['Cluster_Label_B'].apply(lambda x: df[df['Cluster_Label_B'].isin(x)]['best_k_value'].nunique()).reset_index(name='Unique_K_Values')

# # Merge the two dataframes
# plot_data = pd.merge(cluster_a_sizes, k_value_counts, on='Cluster_Label_A')

# # Scale point sizes based on Cluster_Size (adjust scaling factor as needed)
# point_sizes = plot_data['Cluster_Size'] * 50

# # Create a scatter plot
# plt.figure(figsize=(10, 6))
# # Plot main circles (scaled by cluster size)
# plt.scatter(plot_data['Cluster_Size'], plot_data['Unique_K_Values'], s=point_sizes, alpha=0.6, c='teal', edgecolors='black', label='Cluster A')
# # Plot center markers (small black dots)
# plt.scatter(plot_data['Cluster_Size'], plot_data['Unique_K_Values'], s=10, c='black', marker='.', label='Center')
# plt.xlabel('Cluster Size (Cluster_Label_A)')
# plt.ylabel('Number of Unique best_k_value (from Cluster_Label_B)')
# plt.title('Cluster Size of A vs. Number of Unique K Values from B')
# plt.grid(True, linestyle='--', alpha=0.7)
# plt.legend()
# plt.tight_layout()
# plt.show()

In [28]:
# # Step 1: Group by Cluster_Label_A and check if all best_k_value are identical
# k_value_counts = df.groupby('Cluster_Label_A').agg({
#     'best_k_value': lambda x: x.nunique(),  # Count unique k-values
#     'Cluster_Label_A': 'count'  # Count occurrences (cluster size)
# }).rename(columns={'Cluster_Label_A': 'Cluster_Size', 'best_k_value': 'Unique_K_Values'}).reset_index()

# # Step 2: Filter for Cluster_Label_A where all best_k_value are identical (Unique_K_Values == 1)
# perfect_match_clusters = k_value_counts[k_value_counts['Unique_K_Values'] == 1]

# # Step 3: Merge with the actual best_k_value for visualization (to color points)
# k_values = df.groupby('Cluster_Label_A')['best_k_value'].first().reset_index()
# plot_data = pd.merge(perfect_match_clusters, k_values, on='Cluster_Label_A')

# # Step 4: Create a scatter plot
# plt.figure(figsize=(10, 6))
# scatter = plt.scatter(
#     plot_data['Cluster_Size'], 
#     plot_data['Unique_K_Values'], 
#     s=plot_data['Cluster_Size'] * 50,  # Scale point size by cluster size
#     c=plot_data['best_k_value'],      # Color by best_k_value
#     cmap='viridis', 
#     alpha=0.6
# )
# plt.colorbar(label='best_k_value')
# plt.xlabel('Cluster Size (Cluster_Label_A)')
# plt.ylabel('Number of Unique best_k_value (from Cluster_Label_B)')
# plt.title('Cluster Size of A vs. Unique K Values (Perfect Matches Only)')
# plt.axhline(y=1, color='gray', linestyle='--', alpha=0.5)  # Highlight y=1
# plt.grid(True, linestyle='--', alpha=0.7)
# plt.tight_layout()
# plt.show()

In [29]:
df=df_result
# Rename column if needed
df = df.rename(columns={'unique_identity': 'best_k_value'})

# Step 1: Count elements per Cluster_Label_A
cluster_counts = df['Cluster_Label_A'].value_counts().to_dict()

# Step 2: Filter rows where best_k_value >= cluster size
mask = df.apply(lambda row: row['best_k_value'][0] >= cluster_counts[row['Cluster_Label_A']], axis=1)

# Step 3: Drop those rows
df_filtered = df[~mask].reset_index(drop=True)


In [30]:
df_filtered=df_filtered.rename(columns={'File_Path_A': 'File_Path'})
df_filtered=df_filtered.drop(columns=['File_Path_B'])


In [31]:
df_filtered.to_csv("ssdeep/AVG_heuristic_Th60.csv")